# Iraq Farms Health: copy the published crop map from Drive to Cloudflare R2

Copies `publish/2026/` from Drive to the R2 bucket `ifh-data`, served at **https://data.iraqfarmshealth.com/crops/2026/**. Only new or changed files are uploaded (compared by MD5), and files no longer in the publish folder are deleted from that one R2 folder; nothing else in the bucket is touched. Plain Python with `boto3` (Amazon's S3 library, which R2 speaks), installed with `pip` in step 2. Run it **after** the publish notebook and the 2021 comparison.

**Keys (once).** In Cloudflare, go to R2 Object Storage → *Manage API tokens* → *Create Account API token*: permission **Object Read & Write**, applied to **ifh-data only**. Copy the three values it shows into Colab's **🔑 Secrets** panel (left bar) with these names, and switch *Notebook access* on:
- `R2_ACCESS_KEY_ID`
- `R2_SECRET_ACCESS_KEY`
- `R2_ENDPOINT`: the S3 endpoint, `https://<account id>.r2.cloudflarestorage.com` (without `/ifh-data`)

The keys never appear in the notebook or its output. Never paste them anywhere else.

**Runtime ▸ Run all.** Step 3 only shows what would change unless `GO = True`. Step 4 checks every file in R2 against Drive by size and MD5.

In [ ]:
# 1. Settings and Drive
RUN_NAME    = "iraq_winter2026"
DATA_FOLDER = "2026"          # publish/<DATA_FOLDER>/ in Drive  ->  crops/<DATA_FOLDER>/ in R2
GO          = False           # False: step 3 only previews. True: step 3 uploads and deletes.
BUCKET      = "ifh-data"
PREFIX      = f"crops/{DATA_FOLDER}/"

from google.colab import drive
drive.mount("/content/drive")
import os, json, hashlib, mimetypes
RUN = f"/content/drive/MyDrive/iraq-farms-health/worldcereal/{RUN_NAME}"
SRC = f"{RUN}/publish/{DATA_FOLDER}"
idx = json.load(open(f"{SRC}/index.json"))
local = {}
for d, _, fs in os.walk(SRC):
    for f in fs:
        p = os.path.join(d, f)
        local[os.path.relpath(p, SRC).replace(os.sep, "/")] = p
print(f"Source: {SRC}\n  {len(local)} files, {sum(os.path.getsize(p) for p in local.values()) / 1e6:.1f} MB")
print(f"  index.json: generated {idx.get('generated_utc')}, {len(idx['processed'])} tiles, {len(idx['cells'])} cells")
print(f"  compare_2021.json present: {'compare_2021.json' in local}")
print(f"Destination: {BUCKET}/{PREFIX}  (https://data.iraqfarmshealth.com/{PREFIX})")

In [ ]:
# 2. Connect to R2 with the keys from Colab's Secrets (nothing is printed or written to disk)
%pip install -q boto3
import boto3
from botocore.config import Config
from google.colab import userdata
s3 = boto3.client("s3", endpoint_url=userdata.get("R2_ENDPOINT").rstrip("/"),
                  aws_access_key_id=userdata.get("R2_ACCESS_KEY_ID"),
                  aws_secret_access_key=userdata.get("R2_SECRET_ACCESS_KEY"),
                  region_name="auto", config=Config(retries={"max_attempts": 5}))

def remote_list():
    out = {}
    for page in s3.get_paginator("list_objects_v2").paginate(Bucket=BUCKET, Prefix=PREFIX):
        for o in page.get("Contents", []):
            out[o["Key"][len(PREFIX):]] = (o["Size"], o["ETag"].strip('"'))
    return out

def md5(p):
    h = hashlib.md5()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""): h.update(b)
    return h.hexdigest()

remote = remote_list()
top = sorted({o["Key"].split("/")[0] for o in s3.list_objects_v2(Bucket=BUCKET).get("Contents", [])})
print(f"Bucket reachable. Top level: {top or '(empty)'}; already in {PREFIX}: {len(remote)} files")

In [ ]:
# 3. Compare, then preview (GO = False) or upload (GO = True). Every file goes up in one part, so R2's ETag is its MD5.
TYPES = {".tif": "image/tiff", ".vrt": "application/xml", ".qml": "application/xml", ".json": "application/json",
         ".geojson": "application/geo+json", ".csv": "text/csv", ".md": "text/markdown", ".txt": "text/plain", ".png": "image/png"}
local_md5 = {k: md5(p) for k, p in local.items()}
upload = [k for k in sorted(local) if k not in remote or remote[k][1] != local_md5[k] or remote[k][0] != os.path.getsize(local[k])]
delete = sorted(set(remote) - set(local))
print(f"To upload: {len(upload)} file(s), {sum(os.path.getsize(local[k]) for k in upload) / 1e6:.1f} MB; "
      f"to delete from R2: {len(delete)}; unchanged: {len(local) - len(upload)}")
for k in upload[:15]: print("   +", k)
if len(upload) > 15: print(f"   ... and {len(upload) - 15} more")
for k in delete[:15]: print("   -", k)
if not GO:
    print("\nNothing changed in R2. To upload: set GO = True in step 1, then Run all again.")
else:
    from concurrent.futures import ThreadPoolExecutor
    def put(k):
        ext = os.path.splitext(k)[1].lower()
        with open(local[k], "rb") as f:
            s3.put_object(Bucket=BUCKET, Key=PREFIX + k, Body=f,
                          ContentType=TYPES.get(ext) or mimetypes.guess_type(k)[0] or "application/octet-stream")
        return k
    # index.json last, so a visitor never gets a new index pointing at cells that are not there yet
    first = [k for k in upload if k != "index.json"]
    with ThreadPoolExecutor(8) as ex:
        for i, k in enumerate(ex.map(put, first), 1):
            if i % 20 == 0 or i == len(first): print(f"   uploaded {i}/{len(first)}")
    if "index.json" in upload: put("index.json"); print("   uploaded index.json")
    for i in range(0, len(delete), 1000):
        s3.delete_objects(Bucket=BUCKET, Delete={"Objects": [{"Key": PREFIX + k} for k in delete[i:i + 1000]]})
    print(f"Done: {len(upload)} uploaded, {len(delete)} deleted.")

In [ ]:
# 4. Check every file: R2 against Drive, by size and MD5
remote = remote_list()
bad = [k for k in local if k not in remote or remote[k] != (os.path.getsize(local[k]), local_md5[k])]
extra = sorted(set(remote) - set(local))
print(f"In R2: {len(remote)} files, {sum(s for s, _ in remote.values()) / 1e6:.1f} MB")
if not GO:
    print("(Preview run: R2 has not been changed, so differences are expected until you run with GO = True.)")
if not bad and not extra:
    print("ALL FILES MATCH")
else:
    print(f"DIFFERENCES: {len(bad)} missing or different, {len(extra)} extra in R2")
    for k in (bad + extra)[:20]: print("  ", k)

In [ ]:
# 5. Optional, once: remove the test folder made while setting up the bucket
REMOVE_TEST_FOLDER = False
if REMOVE_TEST_FOLDER:
    keys = [o["Key"] for o in s3.list_objects_v2(Bucket=BUCKET, Prefix="test/").get("Contents", [])]
    if keys: s3.delete_objects(Bucket=BUCKET, Delete={"Objects": [{"Key": k} for k in keys]})
    print("removed:", keys)